## Modern Embedding : BERT

BERT (Bidirectional Encoder Representations from Transformers) est un modèle de traitement automatique du langage (NLP) développé par Google en 2018. Ce modèle préentraîné révolutionne la compréhension du langage naturel en capturant le contexte des mots dans les deux directions d’une phrase, améliorant ainsi la précision des systèmes d’analyse linguistique. Il est devenu une référence pour de nombreuses tâches de NLP telles que la classification, la traduction, la recherche ou le question-réponse.

In [1]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

/home/aea/.pyenv/versions/MHSD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA n'est pas disponible. Verifiez les drivers NVIDIA, la presence du GPU et l'environnement du notebook."
    )

device = torch.device("cuda")
print(f"Device utilise: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Device utilise: cuda
GPU: NVIDIA GeForce RTX 2070 Super with Max-Q Design


In [4]:
# Ouvre le CSV ../data/reddit_depression_dataset_cleaned_v2.csv
df = pd.read_csv("../data/reddit_depression_dataset_cleaned_v2.csv")

In [5]:
df.head()

,subreddit,upvotes,created_utc,num_comments,label,clean_text
0,DeepThoughts,4.0,1.405309e+09,NaN,0.0,deep thoughts underdog only when we start cons...
1,DeepThoughts,4.0,1.410568e+09,1.0,0.0,i like this sub theres only two posts yet i ke...
2,DeepThoughts,6.0,1.416458e+09,1.0,0.0,rebirth hello \ni am the new guy in charge her...
3,DeepThoughts,25.0,1.416512e+09,2.0,0.0,i want to be like water i want to slip through...
4,DeepThoughts,5.0,1.416516e+09,4.0,0.0,who am i you could take any one cell in my bod...


In [6]:
# Prépare les données pour l'entraînement
texts = df['clean_text'].tolist()
labels = df['label'].tolist()

In [7]:
# 1. Split (même que LogisticRegression)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, 
    test_size=0.2, random_state=42, stratify=labels
)

In [9]:
# modele BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
model.to(device)
model.eval()

def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def embed(texts, batch_size=32):
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_raw = texts[i:i + batch_size]
            batch_texts = [normalize_text(t) for t in batch_raw]
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=512
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)

            # moyenne des embeddings
            last_hidden = outputs.last_hidden_state
            emb = last_hidden.mean(dim=1).cpu().numpy()
            embeddings.append(emb)

    return np.vstack(embeddings)

# embeddings
X_train_emb = embed(X_train)
X_test_emb = embed(X_test)

# modele classique dessus
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, y_train)

# evaluation
y_pred = clf.predict(X_test_emb)
print(classification_report(y_test, y_pred))

KeyboardInterrupt: 